# PhysAR-IDS -- NB4h (v6): the three derived electrical features under attack -- do they confer immunity?

**The features.** `voltage_imbalance` = max|V - mean V| / mean V over the three phases; `current_imbalance` likewise for the currents; `power_factor` = P / sqrt(P^2 + Q^2). They are computed per row from the nine measurements (V_A..C, I_A..C, P, Q), so an attacker who forges readings *consistently* changes them in a predictable way, and one who perturbs them *independently* (as the all-93-feature attacks do) can make them inconsistent with the readings.

**One run, four questions.**
1. **Behavior.** For each attack, how do the three features move: share of rows changed, size of the shift (in training standard deviations), share pushed outside their genuine range (0.5th to 99.5th percentile of the training rows), and the clean versus attacked raw values.
   * **B** electrical attacker, 5% of each reading, power balance kept, features recomputed (NB4d / NB4f rules);
   * **C** the same plus 0.5 sigma on the 60 cyber features;
   * **F** all 93 features perturbed *independently* by 0.1 sigma (the all-feature attack of NB3c), so the derived features are not tied to the readings.
2. **Integrity check.** A check that recomputes the three features from the readings and compares (and a check on the power-balance ratio). How many attacked rows, and how many *evaded* rows, does it flag, and what is evasion once the check is added? On clean rows it should flag nothing.
3. **Reliance.** TreeSHAP share of the three features in the true-class margin, clean versus attacked, and the signed shift toward benign they carry; and a counterfactual that restores each feature (and all three) to its original value in the evaded rows.
4. **Immunity by ablation.** Three classifiers per seed: full, without the three derived features, without all 33 electrical features. Each faces the same cyber-only white-box attacker (0.1 and 0.5 sigma). If the features gave immunity, removing them would raise evasion. Burst-clustered intervals and paired tests for the differences; clean weighted F1 for each model.

**Rows.** A fixed stratified subsample of the held-out attack rows (`N_PER_CLASS` per class). Attack, constraints, power balance, frozen temporal features and seeds are as in NB4f/NB4g. Removed features are replaced by a constant column (the trees cannot split on it), which is equivalent to removal except that the column sampler still draws it.

**Needs** `phase1_data_v6.pkl`, `standard_scaler_v6.pkl`, `xgb_tuned_best_params_v6.json`, `optuna_xgboost_results_v6.pkl`. Does not need NB4d's files. **Runtime:** about 1 to 2 hours in total for five seeds (setting C and the nine models dominate). Resumable per seed.

In [ ]:
import os, sys, time, json, hashlib, warnings
import numpy as np, pandas as pd, joblib
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def find_project_dir(env_names=('PHYSAR_DIR',), marker='phase1_data_v6.pkl'):
    """Locate the folder that holds the pipeline files, wherever the notebook is run from.
    Order: environment variable(s) -> the current folder and its parents (and a SmartGrid_Dissertation folder next to or inside each) -> the usual places under the home folder."""
    from pathlib import Path
    cands = []
    for n in env_names:
        if os.environ.get(n): cands.append(Path(os.environ[n]).expanduser())
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents][:6]:
        cands += [p, p / 'SmartGrid_Dissertation'] + [p / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    home = Path.home()
    for extra in ('/content', '/content/drive/MyDrive', '/content/drive/MyDrive/smartgrid1', '/content/drive/MyDrive/smartgrid1/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1', '/content/drive/MyDrive/SmartGrid_Dissertation', '/content/drive/MyDrive/smartrid1/SmartGrid_Dissertation', '/content/drive/My Drive', '/content/drive/My Drive/smartgrid1', '/content/drive/My Drive/smartgrid1/SmartGrid_Dissertation', '/content/drive/My Drive/smartrid1/SmartGrid_Dissertation', '/kaggle/working'):
        cands += [Path(extra), Path(extra) / 'SmartGrid_Dissertation']
    for base in ('Desktop', 'Documents', 'Downloads', 'OneDrive', 'OneDrive/Desktop', 'OneDrive/Documents', ''):
        b = home / base if base else home
        cands += [b / 'SmartGrid_Dissertation'] + [b / n / q for n in ('smartgrid1', 'smartrid1') for q in ('', 'SmartGrid_Dissertation')]
    seen = set()
    for c in cands:
        try: c = c.resolve()
        except OSError: continue
        if c in seen: continue
        seen.add(c)
        if (c / marker).is_file(): return str(c)
    for root in [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')] + [home / b for b in ('Desktop', 'Documents', 'Downloads', 'OneDrive')]:   # shallow search, depth <= 3
        if not root.is_dir(): continue
        for dp, dn, fn in os.walk(root):
            if marker in fn: return dp
            if Path(dp).relative_to(root).parts.__len__() >= 3: dn[:] = []
    raise FileNotFoundError(f"Could not find {marker}. Set the environment variable PHYSAR_DIR to the folder that holds the pipeline files "
                            f"(e.g. in a cell, before running: os.environ['PHYSAR_DIR'] = r'C:\\path\\to\\SmartGrid_Dissertation'), or upload the files to the same folder as the notebook (on Colab: /content, or mount Google Drive first).")

PROJECT_DIR = find_project_dir(('NB4H_DIR', 'PHYSAR_DIR')); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = bool(int(os.environ.get('NB4H_SMOKE', '0')))
K_CAND = 12            # nearest split thresholds tried in each direction per quantity (plus the two interval ends)
MAX_ITERS = 60         # greedy rounds (one move per row per round)
CHUNK = 20000          # rows per search block
BLOCK = 400_000        # candidate rows scored per model call
N_PER_CLASS = 500      # attack rows per class evaluated (None = all attack rows)
N_BOOT = 2000
SETTINGS = {                                       # the three attacks whose effect on the derived features is studied
    'B_elec5_balance':         dict(cyber=None, eps_rel=0.05, balance=True),
    'C_cyber0p5_elec5_bal':    dict(cyber=0.5,  eps_rel=0.05, balance=True),
    'F_all93_independent_0p1': dict(cyber=0.1,  eps_rel=0.0,  balance=False, cols='all'),     # every column moved on its own, no consistency
}
T4_BUDGETS = {'0.1': 0.1, '0.5': 0.5}              # cyber-only white-box budgets for the ablation models
T4_SETTINGS = {f'T4_{k}': dict(cyber=v, eps_rel=0.0, balance=True) for k, v in T4_BUDGETS.items()}
ALL_SETTINGS = {**SETTINGS, **T4_SETTINGS}
CONS_TOL = 1e-4                                    # tolerance (in sigma units) of the derived-feature integrity check; clean rows reproduce to ~1e-8
SETTINGS_TO_RUN = list(SETTINGS)
if SMOKE_TEST: SEEDS = SEEDS[:1]; K_CAND = 4; MAX_ITERS = 6; N_PER_CLASS = 60; N_BOOT = 200

os.environ['PYTHONHASHSEED'] = '42'
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'xgboost {xgb.__version__} | search backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"} | smoke {SMOKE_TEST} | working directory: {os.getcwd()}')

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train64 = np.asarray(data['X_train_scaled'], dtype=np.float64); X_test64 = np.asarray(data['X_test_scaled'], dtype=np.float64)
X_train = X_train64.astype(np.float32); X_test = X_test64.astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
scaler = joblib.load(f'standard_scaler_{PIPELINE_VERSION}.pkl'); MU = np.asarray(scaler.mean_, dtype=np.float64); SC = np.asarray(scaler.scale_, dtype=np.float64)
assert len(MU) == N_FEAT

atk_te = y_test != 0; X_atk_test64 = X_test64[atk_te]; y_atk_test = y_test[atk_te]
FEAT_LO, FEAT_HI = X_train.min(0).astype(np.float64), X_train.max(0).astype(np.float64)
PB_NAMES = ['V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower', 'Theta']
PB = np.array([FEATURE_COLS.index(c) for c in PB_NAMES]); DER = np.array([FEATURE_COLS.index(c) for c in ['voltage_imbalance', 'current_imbalance', 'power_factor']])
N_PHYS = len(PB)

def raw_of(Xs): return Xs[:, PB] * SC[PB] + MU[PB]
def derive(raw):
    V, I = raw[:, 0:3], raw[:, 3:6]; Vm, Im = V.mean(1), I.mean(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vimb = np.abs(V - Vm[:, None]).max(1) / np.where(Vm == 0, np.nan, Vm); cimb = np.abs(I - Im[:, None]).max(1) / np.where(Im == 0, np.nan, Im)
        S = np.hypot(raw[:, 6], raw[:, 7]); pf = raw[:, 6] / np.where(S == 0, np.nan, S)
    return np.nan_to_num(np.stack([vimb, cimb, pf], 1))
def ratio_of(raw):
    VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1)
    with np.errstate(divide='ignore', invalid='ignore'): return np.hypot(raw[:, 6], raw[:, 7]) / np.where(VI == 0, np.nan, VI)
def write_back(x, raw): x[:, PB] = (raw - MU[PB]) / SC[PB]; x[:, DER] = (derive(raw) - MU[DER]) / SC[DER]

raw_train = raw_of(X_train64); RAW_LO, RAW_HI = raw_train.min(0), raw_train.max(0)
r_tr = ratio_of(raw_train); r_tr = r_tr[np.isfinite(r_tr)]; BAND_LO, BAND_HI = np.percentile(r_tr, [0.5, 99.5])
r_te = ratio_of(raw_of(X_test64)); r_te = r_te[np.isfinite(r_te)]
der_err = np.abs((derive(raw_of(X_test64)) - MU[DER]) / SC[DER] - X_test64[:, DER]).max()
print(f'Features {N_FEAT} | cyber {N_CYBER} | physical channel {N_PHYS} | attack rows in the test period {len(X_atk_test64):,}')
print(f'Power balance on the test rows: 1st {np.percentile(r_te, 1):.5f}, median {np.median(r_te):.5f}, 99th {np.percentile(r_te, 99):.5f} | genuine band [{BAND_LO:.5f}, {BAND_HI:.5f}] | max derived-feature recomputation error {der_err:.1e}')
if not SMOKE_TEST:
    assert abs(np.median(r_te) - 1.0) < 1e-3 and np.percentile(r_te, 1) > 0.998 and np.percentile(r_te, 99) < 1.002, 'power balance not reproduced -- scaler / column order problem'
    assert der_err < 1e-3, 'derived-feature recomputation does not reproduce the stored values'
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
rng0 = np.random.RandomState(0)
pick = (np.arange(len(y_atk_test)) if N_PER_CLASS is None else np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), min(N_PER_CLASS, int((y_atk_test == c).sum())), replace=False) for c in range(1, 6)])))
X_wb, y_wb, burst_wb = X_atk_test64[pick], y_atk_test[pick], test_burst[pick]
print(f'Evaluation rows: {len(pick):,} ({np.bincount(y_wb)[1:].tolist()} per class) in {len(np.unique(burst_wb))} of {len(np.unique(test_burst))} attack bursts')
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1) if len(nb2_tuned) > 1 else 0:.4f}')

In [ ]:
def get_cols(st): return np.arange(N_FEAT) if st.get('cols') == 'all' else CYBER_IDX
def get_thresholds(clf):
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}
class Margin:
    def __init__(self, clf): self.b = clf.get_booster()
    def __call__(self, X):
        out = []
        if len(X) == 0: return np.zeros((0, 6))
        for s in range(0, len(X), BLOCK):
            out.append(np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + BLOCK], dtype=np.float32), predict_type='margin', validate_features=False)))
        return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)        # > 0  <=>  benign is the arg-max class

def make_ctx(x0):
    raw0 = raw_of(x0); rho0 = ratio_of(raw0); return raw0, np.where(np.isfinite(rho0), rho0, 1.0)
def finalize(x, raw, rho0, balance):
    """x (m, 93) float64 modified in place from the (m, 9) forged raw measurements: optional power-balance rescaling of (P, Q), then write-back with recomputed derived features."""
    if balance:
        VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1); S = np.hypot(raw[:, 6], raw[:, 7])
        f = np.where(S > 0, rho0 * VI / np.where(S > 0, S, 1.0), 1.0); f = np.where(np.isfinite(f) & (f > 0), f, 1.0); raw[:, 6] *= f; raw[:, 7] *= f
    write_back(x, raw)
def phys_interval(raw0, eps):
    a, b = raw0 * (1 - eps), raw0 * (1 + eps); lo, hi = np.minimum(a, b), np.maximum(a, b)
    return np.maximum(lo, np.minimum(RAW_LO, raw0)), np.minimum(hi, np.maximum(RAW_HI, raw0))

def cand_z(xz, a, b, T, K):
    """candidate values (standardized) for one quantity: the K nearest thresholds on each side inside [a, b], plus the interval ends. Returns (row index, value)."""
    nT = len(T); idx = np.searchsorted(T, xz, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)].astype(np.float64); ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)).astype(np.float64); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xz; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xz; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V)

def one_step(margin, x, x0, raw0, rho0, g, st, TH, K):
    n = len(x); best_g = g.copy(); best_kd = np.full(n, -1, np.int8); best_j = np.zeros(n, np.int64); best_v = np.zeros(n); bufs = []; size = [0]; eps = st['eps_rel']
    lo_raw, hi_raw = phys_interval(raw0, eps)
    def flush():
        if not bufs: return
        R = np.concatenate([b[0] for b in bufs]); KD = np.concatenate([b[1] for b in bufs]); J = np.concatenate([b[2] for b in bufs]); VV = np.concatenate([b[3] for b in bufs]); bufs.clear(); size[0] = 0
        for s in range(0, len(R), BLOCK):
            r, kd, jj, v = R[s:s + BLOCK], KD[s:s + BLOCK], J[s:s + BLOCK], VV[s:s + BLOCK]; Xc = x[r].copy()
            m = np.flatnonzero(kd == 0)
            if len(m): Xc[m, jj[m]] = v[m]
            m = np.flatnonzero(kd == 1)
            if len(m):
                raw = raw_of(Xc[m]); raw[np.arange(len(m)), jj[m]] = v[m]; sub = Xc[m]; finalize(sub, raw, rho0[r[m]], st['balance']); Xc[m] = sub
            gg = gap_of(margin(Xc.astype(np.float32)))
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_kd[rr] = kd[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    def add(R, kd, j, V):
        if len(R) == 0: return
        bufs.append((R, np.full(len(R), kd, np.int8), np.full(len(R), j, np.int64), V)); size[0] += len(R)
        if size[0] >= BLOCK: flush()
    for jm in (range(N_PHYS) if eps > 0 else []):                                 # physical measurements (none when the readings are fixed)
        col = int(PB[jm]); T = TH.get(col)
        if T is None: continue
        a = (lo_raw[:, jm] - MU[col]) / SC[col]; b = (hi_raw[:, jm] - MU[col]) / SC[col]
        R, Z = cand_z(x[:, col], a, b, T, K)
        add(R, 1, jm, Z * SC[col] + MU[col])
    if st['cyber'] is not None:
        B = st['cyber']
        for col in get_cols(st):
            T = TH.get(int(col))
            if T is None: continue
            o = x0[:, col]; a = np.maximum(o - B, np.minimum(FEAT_LO[col], o)); b = np.minimum(o + B, np.maximum(FEAT_HI[col], o))
            R, Z = cand_z(x[:, col], a, b, T, K); add(R, 0, int(col), Z)
    flush(); return best_g, best_kd, best_j, best_v

def apply_moves(x, ids, kd, jj, vv, rho0, st):
    m = np.flatnonzero(kd == 0)
    if len(m): x[ids[m], jj[m]] = vv[m]
    m = np.flatnonzero(kd == 1)
    if len(m):
        sub = x[ids[m]]; raw = raw_of(sub); raw[np.arange(len(m)), jj[m]] = vv[m]; finalize(sub, raw, rho0[ids[m]], st['balance']); x[ids[m]] = sub

def whitebox(margin, X0, st, TH, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    x = X0.copy(); raw0, rho0 = make_ctx(X0); g = gap_of(margin(x.astype(np.float32))); init_evaded = g > 0; active = ~init_evaded; stalled = np.zeros(len(x), bool); log = []
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time()
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, kd, jj, vv = one_step(margin, x[ids], X0[ids], raw0[ids], rho0[ids], g[ids], st, TH, K)
            imp = kd >= 0; ii = ids[imp]; apply_moves(x, ii, kd[imp], jj[imp], vv[imp], rho0, st)
            g[ii] = gap_of(margin(x[ii].astype(np.float32))) if len(ii) else g[ii]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); log.append((it + 1, len(act), time.time() - t0))
        if verbose: print(f'      round {it + 1:>2}: active rows {len(act):>6} | {time.time() - t0:5.1f}s | evaded so far {(g > 0).mean():.3f}', flush=True)
    return x, g, init_evaded, stalled, log
print('white-box attack defined')

In [ ]:
atk_names = class_names[1:]

## 3. Self-test (first seed)

One-step search against an independent enumeration, then constraint checks for **every** attack used here: budgets, power balance (where enforced), frozen features, electrical readings untouched in the cyber-only ablation attacks, monotone margin.

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test64), 40, replace=False); X0 = X_atk_test64[rows]; raw0, rho0 = make_ctx(X0); g0 = gap_of(margin(X0.astype(np.float32)))
    for sname in ('B_elec5_balance',):
        st = ALL_SETTINGS[sname]; bg, bkd, bj, bv = one_step(margin, X0.copy(), X0, raw0, rho0, g0, st, TH, K=10 ** 6); lo_raw, hi_raw = phys_interval(raw0, st['eps_rel']); worst = 0.0
        wide_gap = 0.0
        for i in range(len(rows)):
            best = g0[i]; best_wide = g0[i]
            for jm in range(N_PHYS):
                col = int(PB[jm]); T = TH.get(col)
                if T is None: continue
                a, b = (lo_raw[i, jm] - MU[col]) / SC[col], (hi_raw[i, jm] - MU[col]) / SC[col]; xz = X0[i, col]; Tf = T.astype(np.float64); Tdn = np.nextafter(T, np.float32(-np.inf)).astype(np.float64)
                fam = np.concatenate([[b] if b > xz else [], [a] if a < xz else [], Tf[(Tf > xz) & (Tf <= b)], Tdn[(Tf <= xz) & (Tdn >= a)]])          # exactly the attack's move family
                wide = np.concatenate([fam, Tf[(Tf >= a) & (Tf <= b)], Tdn[(Tdn >= a) & (Tdn <= b)]])                                               # plus same-cell points
                for pts, is_fam in ((fam, True), (wide, False)):
                    for z in np.unique(pts):
                        xr = X0[i:i + 1].copy(); raw = raw_of(xr); raw[0, jm] = z * SC[col] + MU[col]; finalize(xr, raw, rho0[i:i + 1], st['balance']); v = gap_of(margin(xr.astype(np.float32)))[0]
                        if is_fam: best = max(best, v)
                        best_wide = max(best_wide, v)
            wide_gap = max(wide_gap, best_wide - bg[i]); worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, {sname}): {bg[i]} < {best}'
        print(f'  {sname}: one_step matches the independent enumeration (largest shortfall {max(worst, 0):.2e}); extra same-cell points would add at most {max(wide_gap, 0):.2f} margin (heuristic gap, evasion stays a lower bound)')
    sub = X_atk_test64[rng.choice(len(X_atk_test64), 300, replace=False)]
    for sname, st in ALL_SETTINGS.items():
        xa, ga, ie, stl, lg = whitebox(margin, sub, st, TH, K=K_CAND, max_iters=4, verbose=False); r0, p0 = make_ctx(sub); raw_f = raw_of(xa)
        cols = get_cols(st) if st['cyber'] is not None else np.array([], int)
        if st['eps_rel'] > 0:
            ix = [0, 1, 2, 3, 4, 5, 8]; rel = np.nan_to_num(np.abs(raw_f[:, ix] / np.where(r0[:, ix] == 0, np.nan, r0[:, ix]) - 1)); assert rel.max() <= st['eps_rel'] + 1e-9, 'relative budget violated'
            assert np.nanmax(np.abs(ratio_of(raw_f) / p0 - 1)) < 1e-9, 'power balance not kept'
        allowed = np.concatenate([cols] + ([PB, DER] if st['eps_rel'] > 0 else [])).astype(int); free = np.setdiff1d(np.arange(N_FEAT), allowed)
        if len(free): assert np.abs(xa[:, free] - sub[:, free]).max() == 0, f'a frozen feature changed ({sname})'
        if st['cyber'] is not None: assert np.abs(xa[:, cols] - sub[:, cols]).max() <= st['cyber'] + 1e-9, 'cyber budget violated'
        if sname.startswith('T4'): assert np.abs(xa[:, PB] - sub[:, PB]).max() == 0 and np.abs(xa[:, DER] - sub[:, DER]).max() == 0, 'electrical readings changed in a cyber-only attack'
        assert (ga >= gap_of(margin(sub.astype(np.float32))) - 1e-9).all(), 'margin gap decreased'
    print('  constraints hold for every attack: budgets, power balance (where enforced), frozen features, untouched readings in the cyber-only attacks, monotone margin')
print('self-test defined')

## 4. Diagnostics, models and the per-seed run (resumable)

In [ ]:
ELEC_IDX = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); DER_NAMES = ['voltage_imbalance', 'current_imbalance', 'power_factor']; MODELS = {'full': np.array([], int), 'no_derived3': DER, 'no_electrical33': ELEC_IDX}
LO_P, HI_P = np.percentile(X_train64[:, DER], [0.5, 99.5], axis=0)
def contrib(booster, X):
    c = np.asarray(booster.predict(xgb.DMatrix(np.ascontiguousarray(X, dtype=np.float32)), pred_contribs=True)); assert c.ndim == 3 and c.shape[1] == 6 and c.shape[2] == N_FEAT + 1, f'unexpected contribution shape {c.shape}'
    return c[:, :, :N_FEAT]
def consistency(xa):
    """integrity checks on a final row: (derived features disagree with the readings, power-balance ratio outside the genuine band)"""
    raw = raw_of(xa); mism = np.abs((derive(raw) - MU[DER]) / SC[DER] - xa[:, DER]).max(1); r = ratio_of(raw); band = ~((r >= BAND_LO) & (r <= BAND_HI)) & np.isfinite(r)
    return mism > CONS_TOL, band, mism
def diagnose(xa, X0, cpu, y, ie):
    ev = cpu.predict(xa.astype(np.float32)) == 0; n = len(xa); ar = np.arange(n); d = xa[:, DER] - X0[:, DER]; ad = np.abs(d); out = {'evaded': ev.copy(), 'clean_benign_rate': float(ie.mean())}
    cons, band, mism = consistency(xa); flag = cons | band; out.update({'cons_flag': cons, 'band_flag': band, 'flag_any': flag, 'evasion_after_checks': float((ev & ~flag).mean()), 'cons_rate_all': float(cons.mean()), 'cons_rate_evaded': float(cons[ev].mean()) if ev.any() else float('nan'),
                                                                     'band_rate_evaded': float(band[ev].mean()) if ev.any() else float('nan'), 'median_mismatch': float(np.median(mism))})
    out['per_class_evasion'] = np.array([ev[y == c].mean() for c in range(1, 6)]); out['overall_evasion'] = float(ev.mean())
    out['changed_share'] = (ad > 0).mean(0); out['abs_delta_mean'] = ad.mean(0); out['abs_delta_median'] = np.median(ad, 0); out['abs_delta_p95'] = np.percentile(ad, 95, axis=0)
    out['outside_range_share'] = ((xa[:, DER] < LO_P) | (xa[:, DER] > HI_P)).mean(0); out['outside_range_clean'] = ((X0[:, DER] < LO_P) | (X0[:, DER] > HI_P)).mean(0)
    out['raw_median_clean'] = np.median(X0[:, DER] * SC[DER] + MU[DER], 0); out['raw_median_adv'] = np.median(xa[:, DER] * SC[DER] + MU[DER], 0)
    out['abs_delta_median_pc'] = np.stack([np.median(ad[y == c], 0) for c in range(1, 6)]); out['changed_share_pc'] = np.stack([(ad[y == c] > 0).mean(0) for c in range(1, 6)])
    out['abs_delta_median_evaded'] = np.median(ad[ev], 0) if ev.any() else np.full(3, np.nan); out['changed_share_evaded'] = (ad[ev] > 0).mean(0) if ev.any() else np.full(3, np.nan)
    cl = contrib(cpu.get_booster(), X0); at = contrib(cpu.get_booster(), xa); yt = y
    for tag, c in (('clean', cl), ('adv', at)): a = np.abs(c[ar, yt]); out[f'share_der_{tag}'] = float((a[:, DER].sum(1) / a.sum(1)).mean())
    dG = (at[:, 0] - at[ar, yt]) - (cl[:, 0] - cl[ar, yt]); out['dG_der'] = dG[:, DER].mean(0); out['dG_der_evaded'] = dG[ev][:, DER].mean(0) if ev.any() else np.full(3, np.nan)
    ret = {}
    for k, nm in enumerate(DER_NAMES):
        xr = xa.copy(); xr[:, DER[k]] = X0[:, DER[k]]; ret[nm] = float((cpu.predict(xr.astype(np.float32))[ev] == 0).mean()) if ev.any() else float('nan')
    xr = xa.copy(); xr[:, DER] = X0[:, DER]; ret['all_three'] = float((cpu.predict(xr.astype(np.float32))[ev] == 0).mean()) if ev.any() else float('nan'); out['restore_retained'] = ret
    return out
def fit_variant(seed, zero):
    Xt = X_train.copy(); Xt[:, zero] = 0.0 if len(zero) else Xt[:, zero]
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(Xt, y_train, verbose=False); tmp = f'_tmp_xgb4h_seed{seed}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu
R = {}; INFO = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack2h_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((best_params, K_CAND, MAX_ITERS, seed, N_PER_CLASS, len(pick), float(X_wb[:5].sum()), [(k, sorted((a_, str(b_)) for a_, b_ in ALL_SETTINGS[k].items())) for k in ALL_SETTINGS], CONS_TOL, 'nb4h-v1'))
    R[seed] = {'diag': {}, 't4': {}}; INFO[seed] = {}
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed], INFO[seed] = saved['R'], saved['info']
    todo = [k for k in SETTINGS if k not in R[seed]['diag']] + [f'{m}|{b}' for m in MODELS for b in T4_BUDGETS if f'{m}|{b}' not in R[seed]['t4']]
    if not todo: print(f'seed {seed}: all found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}: {len(todo)} item(s) to run', flush=True)
    clf, cpu = fit_variant(seed, np.array([], int)); margin = Margin(clf); TH = get_thresholds(clf); INFO[seed]['full'] = {'clean_weighted_f1': float(f1_score(y_test, cpu.predict(X_test), average='weighted'))}
    print(f'  full model: clean weighted F1 {INFO[seed]["full"]["clean_weighted_f1"]:.4f}', flush=True)
    if seed == SEEDS[0] and not R[seed]['diag'] and not R[seed]['t4']: selftest(clf, margin, TH)
    for sname in [k for k in SETTINGS if k in todo]:
        st = SETTINGS[sname]; ts = time.time(); print(f'  [{sname}] attacking {len(X_wb)} rows ...', flush=True)
        xa, ga, ie, stl, lg = whitebox(margin, X_wb, st, TH, verbose=False); d = diagnose(xa, X_wb, cpu, y_wb, ie); d['rows_stalled'] = float((stl & ~d['evaded']).mean()); d['rows_iter_limit'] = float((~stl & ~d['evaded']).mean()); R[seed]['diag'][sname] = d
        print(f'  [{sname}] evasion {d["overall_evasion"]:.3f} per class {np.round(d["per_class_evasion"], 3).tolist()} | features changed (vimb, cimb, pf) {np.round(d["changed_share"], 3).tolist()} | outside genuine range {np.round(d["outside_range_share"], 3).tolist()} | integrity check flags {d["cons_rate_all"]:.3f} of rows ({d["cons_rate_evaded"]:.3f} of evaded), power-balance band flags {d["band_rate_evaded"]:.3f} of evaded | after checks {d["evasion_after_checks"]:.3f} | restore-all-three keeps {d["restore_retained"]["all_three"]:.3f} | {(time.time() - ts) / 60:.1f} min', flush=True)
        joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed]}, pf)
    del clf, cpu, margin
    for mname, zero in MODELS.items():
        need = [b for b in T4_BUDGETS if f'{mname}|{b}' in todo]
        if not need: continue
        clf, cpu = fit_variant(seed, zero); margin = Margin(clf); TH = get_thresholds(clf); f1c = float(f1_score(y_test, cpu.predict(X_test), average='weighted')); INFO[seed][mname] = {'clean_weighted_f1': f1c, 'uses_derived': int(sum(int(c) in TH for c in DER)), 'uses_electrical': int(sum(int(c) in TH for c in ELEC_IDX))}
        print(f'  model {mname}: clean weighted F1 {f1c:.4f} | derived features used by the trees {INFO[seed][mname]["uses_derived"]} of 3, electrical {INFO[seed][mname]["uses_electrical"]} of 33', flush=True)
        for b in need:
            ts = time.time(); xa, ga, ie, stl, lg = whitebox(margin, X_wb, T4_SETTINGS[f'T4_{b}'], TH, verbose=False); ev = cpu.predict(xa.astype(np.float32)) == 0
            R[seed]['t4'][f'{mname}|{b}'] = {'evaded': ev.copy(), 'per_class_evasion': np.array([ev[y_wb == c].mean() for c in range(1, 6)]), 'overall_evasion': float(ev.mean()), 'clean_benign_rate': float(ie.mean())}
            print(f'    cyber-only white-box {b} sigma: evasion {ev.mean():.3f} per class {np.round(R[seed]["t4"][f"{mname}|{b}"]["per_class_evasion"], 3).tolist()} | {(time.time() - ts) / 60:.1f} min', flush=True)
        joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed]}, pf); del clf, cpu, margin
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. How the three features move (mean over seeds; shifts in training standard deviations)

In [ ]:
pd.set_option('display.width', 320); pd.set_option('display.max_colwidth', 90)
def ms(vals): v = np.asarray(vals, float); return f'{np.nanmean(v):.3f} +/- {(np.nanstd(v, ddof=1) if len(v) > 1 else 0):.3f}'
rows = []
for sname in SETTINGS:
    D = [R[s]['diag'][sname] for s in SEEDS]
    for k, nm in enumerate(DER_NAMES):
        rows.append({'attack': sname, 'feature': nm, 'rows changed': round(float(np.mean([d['changed_share'][k] for d in D])), 3), 'median |shift| (sigma)': round(float(np.mean([d['abs_delta_median'][k] for d in D])), 4),
                     'mean |shift| (sigma)': round(float(np.mean([d['abs_delta_mean'][k] for d in D])), 4), '95th pct |shift| (sigma)': round(float(np.mean([d['abs_delta_p95'][k] for d in D])), 4),
                     'outside genuine range: clean -> attacked': f'{np.mean([d["outside_range_clean"][k] for d in D]):.3f} -> {np.mean([d["outside_range_share"][k] for d in D]):.3f}',
                     'median raw value: clean -> attacked': f'{np.mean([d["raw_median_clean"][k] for d in D]):.5f} -> {np.mean([d["raw_median_adv"][k] for d in D]):.5f}',
                     'rows changed, evaded rows only': round(float(np.nanmean([d['changed_share_evaded'][k] for d in D])), 3), 'median |shift|, evaded rows (sigma)': round(float(np.nanmean([d['abs_delta_median_evaded'][k] for d in D])), 4)})
behavior = pd.DataFrame(rows); print(behavior.to_string(index=False))
print('\nPer attack class: median |shift| (sigma) of (voltage_imbalance, current_imbalance, power_factor)')
pcr = []
for sname in SETTINGS:
    a = np.mean([R[s]['diag'][sname]['abs_delta_median_pc'] for s in SEEDS], 0); b = np.mean([R[s]['diag'][sname]['changed_share_pc'] for s in SEEDS], 0)
    for i, c in enumerate(atk_names): pcr.append({'attack': sname, 'class': c, **{f'{nm} median |shift|': round(float(a[i, k]), 4) for k, nm in enumerate(DER_NAMES)}, **{f'{nm} changed': round(float(b[i, k]), 3) for k, nm in enumerate(DER_NAMES)}})
class_behavior = pd.DataFrame(pcr); print(class_behavior.to_string(index=False))

## 6. Integrity check: does recomputing the derived features (and the power-balance ratio) expose the attack?

In [ ]:
rows = []
for sname in SETTINGS:
    D = [R[s]['diag'][sname] for s in SEEDS]
    rows.append({'attack': sname, 'evasion': ms([d['overall_evasion'] for d in D]), 'derived-feature check flags (all attacked rows)': ms([d['cons_rate_all'] for d in D]), 'flags (evaded rows)': ms([d['cons_rate_evaded'] for d in D]),
                 'power-balance band flags (evaded rows)': ms([d['band_rate_evaded'] for d in D]), 'evasion after both checks': ms([d['evasion_after_checks'] for d in D])})
integrity = pd.DataFrame(rows); print(integrity.to_string(index=False))
cl_flag = np.mean([consistency(X_wb)[0].mean() for _ in [0]]); print(f'\nFalse alarms of the derived-feature check on the unperturbed attack rows: {cl_flag:.4f}; of the power-balance band on the same rows: {consistency(X_wb)[1].mean():.4f} (the band is the 0.5th to 99.5th percentile of training rows, so about 0.01 is expected on clean data)')

## 7. Reliance on the three features, and what happens when each is restored (evaded rows)

In [ ]:
rows = []
for sname in SETTINGS:
    D = [R[s]['diag'][sname] for s in SEEDS]
    rows.append({'attack': sname, '|SHAP| share of the 3 features, clean': ms([d['share_der_clean'] for d in D]), 'attacked': ms([d['share_der_adv'] for d in D]),
                 **{f'shift toward benign, {nm}': ms([d['dG_der'][k] for d in D]) for k, nm in enumerate(DER_NAMES)},
                 **{f'still evaded if {nm} restored': ms([d['restore_retained'][nm] for d in D]) for nm in DER_NAMES}, 'still evaded if all three restored': ms([d['restore_retained']['all_three'] for d in D])})
reliance = pd.DataFrame(rows); print(reliance.T.to_string(header=False))

## 8. Immunity by ablation: the same cyber-only white-box attacker against three classifiers

If the three derived features (or the electrical features generally) protected the model against a network attacker, a classifier trained without them would be evaded more. Intervals resample attack bursts (350 in the test period), 2,000 draws, pooled over seeds.

In [ ]:
ub = np.unique(test_burst); bi_full = np.searchsorted(ub, test_burst); nb_ = len(ub); idx_b = np.random.RandomState(42).randint(0, nb_, size=(N_BOOT, nb_)); bi_sub = bi_full[pick]
def bstat(vals, bi, m=None):
    if m is not None: vals = [v[m] for v in vals]; bi = bi[m]
    cnt = np.bincount(bi, minlength=nb_).astype(float); draws = [np.bincount(bi, weights=v.astype(float), minlength=nb_)[idx_b].sum(1) / cnt[idx_b].sum(1) for v in vals]; d = np.mean(draws, 0)
    return float(np.mean([v.mean() for v in vals])), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ci_rows = []
def add_ci(label, vals, m=None): e = bstat(vals, bi_sub, m); ci_rows.append({'quantity': label, 'estimate': round(e[0], 4), 'lo': round(e[1], 4), 'hi': round(e[2], 4)}); return e
for b in T4_BUDGETS:
    for mname in MODELS:
        ev = [R[s]['t4'][f'{mname}|{b}']['evaded'] for s in SEEDS]; add_ci(f'{b} sigma, cyber-only white-box: evasion of {mname}', ev)
    for mname in ('no_derived3', 'no_electrical33'):
        add_ci(f'{b} sigma: {mname} minus full', [R[s]['t4'][f'{mname}|{b}']['evaded'].astype(float) - R[s]['t4'][f'full|{b}']['evaded'] for s in SEEDS])
        for c in range(1, 6): add_ci(f'{b} sigma: {mname} minus full, {atk_names[c - 1]}', [R[s]['t4'][f'{mname}|{b}']['evaded'].astype(float) - R[s]['t4'][f'full|{b}']['evaded'] for s in SEEDS], m=(y_wb == c))
for sname in SETTINGS:
    add_ci(f'{sname}: evasion', [R[s]['diag'][sname]['evaded'] for s in SEEDS]); add_ci(f'{sname}: evasion after integrity checks', [R[s]['diag'][sname]['evaded'] & ~R[s]['diag'][sname]['flag_any'] for s in SEEDS])
    add_ci(f'{sname}: derived-feature check flags (evaded rows count as part of all attacked rows)', [R[s]['diag'][sname]['cons_flag'] for s in SEEDS])
ci_table = pd.DataFrame(ci_rows); print(ci_table.to_string(index=False))
pr = []
for b in T4_BUDGETS:
    for mname in ('no_derived3', 'no_electrical33'):
        a = np.array([R[s]['t4'][f'{mname}|{b}']['overall_evasion'] for s in SEEDS]); f = np.array([R[s]['t4'][f'full|{b}']['overall_evasion'] for s in SEEDS]); dd = a - f
        pr.append({'budget': f'{b} sigma', 'model': mname, 'evasion': a.mean(), 'full model': f.mean(), 'diff': dd.mean(), 'p_raw': 1.0 if dd.std(ddof=1) == 0 and dd.mean() == 0 else float(stats.ttest_rel(a, f).pvalue)})
paired = pd.DataFrame(pr); paired['p_holm'] = holm(paired.p_raw.values); print('\nPaired t-tests over seeds (model variance only):'); print(paired.round(4).to_string(index=False))
cr = [{'model': m, 'clean weighted F1': ms([INFO[s][m]['clean_weighted_f1'] for s in SEEDS]), **({'derived features used (of 3)': round(float(np.mean([INFO[s][m]['uses_derived'] for s in SEEDS])), 1), 'electrical features used (of 33)': round(float(np.mean([INFO[s][m]['uses_electrical'] for s in SEEDS])), 1)} if m != 'full' else {})} for m in MODELS]
clean_table = pd.DataFrame(cr); print('\nClean performance of the three classifiers:'); print(clean_table.to_string(index=False))
pcr = []
for b in T4_BUDGETS:
    for mname in MODELS: pcr.append({'budget': f'{b} sigma', 'model': mname, **{c: ms([R[s]['t4'][f'{mname}|{b}']['per_class_evasion'][i] for s in SEEDS]) for i, c in enumerate(atk_names)}})
t4_class = pd.DataFrame(pcr); print('\nPer-class evasion:'); print(t4_class.to_string(index=False))

## 9. Save

In [ ]:
tables = {'behavior': behavior, 'class_behavior': class_behavior, 'integrity': integrity, 'reliance': reliance.T.reset_index(), 'burst_ci': ci_table, 'paired': paired, 'clean_models': clean_table, 't4_class': t4_class}
for k_, t_ in tables.items(): t_.to_csv(f'attack2h_{k_}_{PIPELINE_VERSION}.csv', index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'K_CAND': K_CAND, 'MAX_ITERS': MAX_ITERS, 'N_PER_CLASS': N_PER_CLASS, 'tables': tables, 'info': INFO,
             'per_seed': {s: {'diag': {k: {kk: vv for kk, vv in v.items() if kk not in ('evaded', 'cons_flag', 'band_flag', 'flag_any')} for k, v in R[s]['diag'].items()}, 't4': {k: {kk: vv for kk, vv in v.items() if kk != 'evaded'} for k, v in R[s]['t4'].items()}} for s in SEEDS}, 'n_bursts': nb_}, f'attack2h_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB4h', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'K_CAND': K_CAND, 'MAX_ITERS': MAX_ITERS, 'N_PER_CLASS': N_PER_CLASS, 'cons_tol': CONS_TOL, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4h_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack2h_results_{PIPELINE_VERSION}.pkl', f'attack2h_behavior_{PIPELINE_VERSION}.csv', f'attack2h_integrity_{PIPELINE_VERSION}.csv', f'attack2h_burst_ci_{PIPELINE_VERSION}.csv', f'nb4h_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)